# 동적 계획법 (Dynamic Programming)

`-` 복잡한 문제를 중복되는 여러 하위 문제로 나눈 뒤 메모이제이션 통해 효율적으로 계산하는 최적화 기법

## 리조트(초등/고등)

- 문제 출처: [정올 2993번](https://jungol.co.kr/problem/2993)

`-` 현재 몇 일인지, 쿠폰이 몇 개 있는지 그리고 연속 이용권에 의해 오늘을 제외하고 며칠 동안 리조트를 무료로 사용 가능한지를 상태 공간으로 나타내고 각 상태 공간별 최소 비용을 저장하면 된다

`-` 매일 $5$일 연속권을 구매해도 쿠폰은 최대 $2N$장 얻을 수 있으니 전체 알고리즘의 시간 복잡도는 최악의 경우에도 $O\left(N^2\right)$이다

`-` 틀려서 보니까 굉장한 반례가 있다

`-` 리조트에 처음으로 갈 수 있는 날이 $N$일을 넘길 수 있는데 이 경우 정답은 $0$이다

In [79]:
def find_first_use_day(close_days):
    day = 1
    while day in close_days:
        day += 1
    return day


def minimize_cost(n, close_days):
    first_use_day = find_first_use_day(close_days)
    if first_use_day > n:
        return 0
    inf = float("inf")
    dp = [[[inf] * 5 for _ in range(2 * n + 1)] for _ in range(n + 1)]
    dp[first_use_day][0][0] = 10000
    dp[first_use_day][1][2] = 25000
    dp[first_use_day][2][4] = 37000
    for day in range(first_use_day, n):
        for num_coupons in range(2 * n + 1):
            nc = num_coupons
            for free in range(5):
                min_cost = dp[day][nc][free]
                if min_cost == inf:
                    continue
                dp[day + 1][nc][max(free - 1, 0)] = min(min_cost + 10000, dp[day + 1][nc][max(free - 1, 0)])
                dp[day + 1][nc + 1][max(free - 1, 2)] = min(min_cost + 25000, dp[day + 1][nc + 1][max(free - 1, 2)])
                dp[day + 1][nc + 2][max(free - 1, 4)] = min(min_cost + 37000, dp[day + 1][nc + 2][max(free - 1, 4)])
                if day + 1 in close_days or free > 0:
                    dp[day + 1][nc][max(free - 1, 0)] = min(min_cost, dp[day + 1][nc][max(free - 1, 0)])
                elif nc >= 3:
                    dp[day + 1][nc - 3][max(free - 1, 0)] = min(min_cost, dp[day + 1][nc - 3][max(free - 1, 0)])
    return min(map(min, dp[n]))


def solution():
    N, M = map(int, input().split())
    close_days = set(map(int, input().split())) if M > 0 else set()
    answer = minimize_cost(N, close_days)
    print(answer)


solution()

# input
# 1 0

 4 0


35000


## 유전자

- 문제 출처: [정올 1701번](https://jungol.co.kr/problem/1701)

`-` $2$차원 DP(사선형)인 문제인데 정확히 무슨 뜻인지는 모르겠다

`-` $\operatorname{dp}[i][j]$를 입력 DNA 서열의 $i$번째 인덱스부터 $j$번째 인덱스까지 고려한 부분 DNA 서열로부터 계산된 최장 KOI 유전자의 길이라고 하자

`-` 조건 (1)에 의해 부분 DNA 서열의 길이가 $2$일 때 at 또는 gc인 경우 최장 KOI 유전자의 길이는 $2$가 되며 그렇지 않은 경우 $0$이 된다

`-` 조건 (1)에 의해 부분 DNA 서열의 길이가 $3$일 때 부분 DNA 서열에서 임의의 문자 하나를 제거한 것이 at 또는 gc인 경우 최장 KOI 유전자의 길이는 $2$가 되며 그렇지 않은 경우 $0$이 된다

`-` 조건 (2)에 의해 $(S[i] =$ a, $S[j] =$ t$)$ 또는 $(S[i] =$ g, $S[j] =$ c$)$인 경우 $\operatorname{dp}[i][j] = 2 + \operatorname{dp}[i + 1][j - 1]$이다 (단, $S$는 입력 DNA 서열)

`-` 만약 그렇지 않은 경우 $S[i]$ 또는 $S[j]$는 최장 KOI 유전자에 포함되지 않을 수 있다 (조건 (3)에 의해 포함될 수도 있음)

`-` 이 경우 $\operatorname{dp}[i][j] = \max(\operatorname{dp}[i + 1][j], \operatorname{dp}[i][j - 1])$이다

`-` 또한, 조건 (3)에 의해 $\operatorname{dp}[i][j] = \max\left(\max\limits_{i < k < j - 1}\{\operatorname{dp}[i][k] + \operatorname{dp}[k + 1][j]\}, \operatorname{dp}[i][j]\right)$이다

`-` 이를 탑다운 형식으로 구현하면 되며 전체 알고리즘의 시간 복잡도는 $O\left(N^3\right)$이다

In [49]:
def find_longest_koi_gene_length(dna):
    n = len(dna)
    inf = float("inf")
    dp = [[-inf] * n for _ in range(n)]
    longest_koi_gene_length = _f(0, n - 1, dna, dp)
    return longest_koi_gene_length


def _f(i, j, dna, dp):
    if dp[i][j] >= 0:
        return dp[i][j]
    n = j - i + 1
    if n < 2:
        dp[i][j] = 0
        return dp[i][j]
    if n == 2:
        if (dna[i] == "a" and dna[j] == "t") or (dna[i] == "g" and dna[j] == "c"):
            dp[i][j] = 2
        else:
            dp[i][j] = 0
        return dp[i][j]
    if (dna[i] == "a" and dna[j] == "t") or (dna[i] == "g" and dna[j] == "c"):
        dp[i][j] = 2 + _f(i + 1, j - 1, dna, dp)
    else:
        dp[i][j] = max(_f(i + 1, j, dna, dp), _f(i, j - 1, dna, dp))
    for k in range(i + 1, j - 1):
        result = _f(i, k, dna, dp) + _f(k + 1, j, dna, dp)
        dp[i][j] = max(result, dp[i][j])
    return dp[i][j]
        

def solution():
    dna = input()
    answer = find_longest_koi_gene_length(dna)
    print(answer)


solution()

# input
# gagtc

 gagtc


4


## 자동차경주대회

- 문제 출처: [정올 1491번](https://jungol.co.kr/problem/1491)

`-` $\operatorname{dp}[i]$를 $i$번 정비소를 방문했고 이를 포함해 여태까지 소요한 정비 시간의 최솟값이라 하자

`-` $T_i$를 $i$번 정비소의 정비 시간이라 하고 $D_i$를 출발 지점으로부터 $i$번 정비소 간의 거리라고 하고 $M$을 정비를 받지 않고 갈 수 있는 최대 거리라고 하자

`-` 그럼 $\operatorname{dp}[i] = T_i + \min\limits_{k<i,\; D_i - D_k \le M}\{\operatorname{dp}[k]\}$를 만족한다

`-` 역추적을 위해 위의 점화식에서 선택된 $k$에 대해 $\operatorname{track}[i] = k$로 정의하자. 이는 $i$번 정비소를 방문하기 전에 $k$번 정비소를 방문했음을 의미한다

`-` 도착 지점까지 가는데 필요한 최소 정비 시간은 $\operatorname{dp}[N + 1]$이다 (단, $N+1$번 정비소는 도착 지점이며 $T_{N+1} = 0$)

`-` 여기에 $\operatorname{track}$ 배열을 토대로 역추적만 해주면 된다

In [18]:
from itertools import accumulate


def solution():
    M = int(input())
    N = int(input())
    D = [0] + [*accumulate(map(int, input().split()))]
    T = [0] + list(map(int, input().split())) + [0]
    inf = float("inf")
    dp = [inf] * (N + 2)
    dp[0] = 0
    track = [None] * (N + 2)
    for i in range(1, N + 2):
        argmin = min([k for k in range(i) if D[i] - D[k] <= M], key=lambda x: dp[x])
        dp[i] = T[i] + dp[argmin]
        track[i] = argmin
    min_duration = dp[N + 1]
    print(min_duration)
    if min_duration == 0:
        return
    order = []
    node = N + 1
    while track[node] > 0:
        order.append(track[node])
        node = track[node]
    print(len(order))
    print(*order[::-1])


solution()

# input
# 111
# 1
# 1 11
# 55

 111
 1
 1 11
 55


0
